# VQA Mutual Cognition Experiment -- Qwen2.5-VL-3B

Agent A sees image -> describes in 3 phrases (token-budgeted)  
Agent B sees description + question + choices -> answers (A/B/C/D)  
4 conditions: blind / a_aware / b_aware / mutual  
Token budgets: [16, 24, 32, 48]  
30 questions from ScienceQA (image subset), seed=42, A caching

In [ ]:
# ============================================================
# Cell 1: Install + Load Qwen2.5-VL-3B
# ============================================================
!pip install -q torch transformers accelerate qwen-vl-utils
!pip install -q datasets  # for ScienceQA

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import torch

print("Loading Qwen2.5-VL-3B-Instruct...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    torch_dtype="auto",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
print(f"Model loaded. Device: {model.device}")

In [ ]:
# ============================================================
# Cell 2: ScienceQA Loader -- filter image questions
# ============================================================
import random, re, time, json

from datasets import load_dataset

print("Loading ScienceQA dataset (test split)...")
ds = load_dataset("derek-thomas/ScienceQA", split="test")
print(f"Loaded {len(ds)} rows")

# Filter for questions with images and >= 2 choices
image_rows = []
for i, row in enumerate(ds):
    img = row.get("image", None)
    if img is not None:
        choices = row.get("choices", [])
        if len(choices) >= 2:
            image_rows.append(row)

print(f"Found {len(image_rows)} questions with images")

# Sample 30
random.seed(42)
samples = random.sample(image_rows, min(30, len(image_rows)))
print(f"Sampled {len(samples)} questions")

# Format
LETTERS = "ABCDEFGH"

def format_question(row):
    choices = row["choices"]
    choices_text = "\n".join(f"{LETTERS[i]}) {c}" for i, c in enumerate(choices))
    answer_idx = row["answer"]  # integer index
    answer_letter = LETTERS[answer_idx]
    return {
        "image": row["image"],      # PIL Image
        "question": row["question"],
        "choices_text": choices_text,
        "answer": answer_letter,
        "n_choices": len(choices),
    }

questions = [format_question(s) for s in samples]
print(f"Ready: {len(questions)} formatted questions")
print(f"Example: {questions[0]['question'][:80]}... Answer={questions[0]['answer']}")

In [ ]:
# ============================================================
# Cell 3: VQA Experiment -- 4 conditions x 4 budgets x 30 Qs
# ============================================================

# ── Prompts ──────────────────────────────────────────────────
A_BLIND = """You are Agent A.

You can inspect the image under these limitations:
- reliable for large objects, actions, and spatial relations
- may miss small objects, text, and subtle details
- report only visible facts; avoid guessing

Describe the image using exactly 3 short phrases:
1. main objects
2. main action or relation
3. visible context

Use at most 15 words total."""

A_AWARE = """You are Agent A.

You can inspect the image under these limitations:
- reliable for large objects, actions, and spatial relations
- may miss small objects, text, and subtle details
- report only visible facts; avoid guessing

You also know the question and answer choices.

Describe the image using exactly 3 short phrases:
1. main objects
2. main action or relation
3. the visible detail most useful for distinguishing the answer choices

Use at most 15 words total.
Do NOT mention any answer choice explicitly."""

B_BLIND = """You are Agent B.

You are given:
- a description from Agent A
- a question
- answer choices

Choose the best answer using the description.

Output ONLY: A, B, C, or D."""

B_AWARE = """You are Agent B.

You are given:
- a description from Agent A
- a question
- answer choices

About Agent A:
- A is reliable for large objects, actions, and spatial relations
- A may miss small objects, text, and subtle details
- A reports visible facts and avoids guessing

Interpret the description with this in mind:
- trust mentioned objects/actions strongly
- do not assume missing small details are absent
- rely on visible relations more than fine detail

Output ONLY: A, B, C, or D."""

# ── Conditions ────────────────────────────────────────────────
CONDITIONS = {
    "blind":   {"a_type": "blind",  "b_knows": False},
    "a_aware": {"a_type": "aware",  "b_knows": False},
    "b_aware": {"a_type": "blind",  "b_knows": True},
    "mutual":  {"a_type": "aware",  "b_knows": True},
}

TX_BUDGETS = [16, 24, 32, 48]

# ── Answer extraction ────────────────────────────────────────
def extract_answer(resp, n_choices=4):
    valid = set(LETTERS[:n_choices])
    t = resp.strip().upper()
    if len(t) == 1 and t in valid:
        return t
    for pat in [
        r'(?:answer|choice)[\s:is]*([A-' + LETTERS[n_choices-1] + r'])\b',
        r'^([A-' + LETTERS[n_choices-1] + r'])[)\.]',
        r'\b([A-' + LETTERS[n_choices-1] + r'])\b',
    ]:
        m = re.search(pat, t, re.IGNORECASE | re.MULTILINE)
        if m and m.group(1).upper() in valid:
            return m.group(1).upper()
    return "N/A"

# ── Inference helpers ─────────────────────────────────────────
def agent_a_vision(system_prompt, pil_image, user_text, max_tokens):
    """Agent A: sees the image. Returns description text."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": [
            {"type": "image", "image": pil_image},
            {"type": "text", "text": user_text},
        ]}
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text], images=[pil_image],
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()


def agent_b_text(system_prompt, user_text, max_tokens=16):
    """Agent B: text-only, no image. Returns answer text."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text}
    ]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor.tokenizer(
        text, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()


# ── Main experiment loop ──────────────────────────────────────
results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}
call_count = 0
t_start = time.time()

print(f"VQA Mutual Cognition Experiment")
print(f"  Model: Qwen2.5-VL-3B-Instruct")
print(f"  Questions: {len(questions)}")
print(f"  Budgets: {TX_BUDGETS}")
print(f"  Conditions: {list(CONDITIONS.keys())}")
print(f"  A caching: blind/b_aware share, a_aware/mutual share")
print()

for qi, q in enumerate(questions):
    print(f"-- Q{qi+1}/{len(questions)}: {q['question'][:60]}...", flush=True)

    for budget in TX_BUDGETS:
        # Cache Agent A responses per budget (blind vs aware)
        a_cache = {}

        for cond_name, cond in CONDITIONS.items():
            a_type = cond["a_type"]    # "blind" or "aware"
            b_knows = cond["b_knows"]  # True or False

            # ── Agent A (vision) ──────────────────────────────
            cache_key = a_type  # same a_type at same budget => reuse
            if cache_key in a_cache:
                a_desc = a_cache[cache_key]
            else:
                if a_type == "blind":
                    a_desc = agent_a_vision(
                        A_BLIND, q["image"],
                        "Describe the image.",
                        max_tokens=budget
                    )
                else:  # aware
                    a_desc = agent_a_vision(
                        A_AWARE, q["image"],
                        f"Question: {q['question']}\nChoices:\n{q['choices_text']}\n\nDescribe the image.",
                        max_tokens=budget
                    )
                a_cache[cache_key] = a_desc
                call_count += 1

            # ── Agent B (text only) ───────────────────────────
            b_system = B_AWARE if b_knows else B_BLIND
            b_input = (
                f"Description:\n{a_desc}\n\n"
                f"Question: {q['question']}\n"
                f"Choices:\n{q['choices_text']}\n\n"
                f"Answer:"
            )
            b_resp = agent_b_text(b_system, b_input, max_tokens=8)
            call_count += 1

            pred = extract_answer(b_resp, q["n_choices"])
            correct = int(pred == q["answer"])
            results[budget][cond_name].append(correct)

        # Progress after each budget
        accs = {c: sum(results[budget][c]) for c in CONDITIONS}
        print(f"   {budget}tok | " + " | ".join(
            f"{c}: {accs[c]}/{qi+1}" for c in CONDITIONS
        ), flush=True)

    # ETA
    elapsed = time.time() - t_start
    eta = elapsed / (qi + 1) * (len(questions) - qi - 1)
    print(f"   [{call_count} calls, {elapsed:.0f}s elapsed, ETA {eta:.0f}s]\n", flush=True)

# ── Results ────────────────────────────────────────────────────
total_time = time.time() - t_start
print(f"\n{'='*60}")
print(f"RESULTS -- {len(questions)} questions, {call_count} calls, {total_time:.0f}s")
print(f"{'='*60}\n")

# Accuracy table
header = f"{'Budget':<10}" + "".join(f"{c:<12}" for c in CONDITIONS)
print(header)
print("-" * len(header))

acc_table = {}
for budget in TX_BUDGETS:
    row = f"{budget}tok{'':<5}"
    for c in CONDITIONS:
        vals = results[budget][c]
        acc = sum(vals) / len(vals) * 100 if vals else 0
        acc_table[(budget, c)] = acc
        row += f"{acc:>5.1f}%{'':<6}"
    print(row)

print()

# ── 2x2 Interaction analysis ──────────────────────────────────
print("2x2 Interaction Analysis (A_aware x B_aware)")
print("-" * 50)

for budget in TX_BUDGETS:
    blind_acc   = acc_table[(budget, "blind")]
    a_aware_acc = acc_table[(budget, "a_aware")]
    b_aware_acc = acc_table[(budget, "b_aware")]
    mutual_acc  = acc_table[(budget, "mutual")]

    a_effect = ((a_aware_acc + mutual_acc) / 2) - ((blind_acc + b_aware_acc) / 2)
    b_effect = ((b_aware_acc + mutual_acc) / 2) - ((blind_acc + a_aware_acc) / 2)
    interaction = mutual_acc - a_aware_acc - b_aware_acc + blind_acc

    print(f"\n  Budget = {budget} tokens:")
    print(f"    blind={blind_acc:.1f}%  a_aware={a_aware_acc:.1f}%  b_aware={b_aware_acc:.1f}%  mutual={mutual_acc:.1f}%")
    print(f"    A-effect: {a_effect:+.1f}pp   B-effect: {b_effect:+.1f}pp   Interaction: {interaction:+.1f}pp")

    if interaction > 2:
        print(f"    >> SUPER-ADDITIVE (synergy)")
    elif interaction < -2:
        print(f"    >> SUB-ADDITIVE (redundancy)")
    else:
        print(f"    >> ADDITIVE")

print(f"\n{'='*60}")
print("Done.")